# DS8 · Comunicación y producto de datos — Soluciones

Un análisis brillante que nadie entiende no cambia ninguna decisión. En este notebook, con los datos de **TiendaSur**, practicas lo que separa a quien «saca números» de quien influye en el negocio:

1. Diseñar **métricas**: métrica norte, árbol de métricas y guardarraíles.
2. **Gráficos** que convencen y gráficos que engañan (con una medida: el factor de mentira).
3. **Storytelling** con datos y un **dashboard** sencillo con Plotly.
4. Un **memo de una página** con la pirámide de Minto y traducir un resultado técnico a lenguaje de negocio.
5. **Make or buy**: construir o comprar, con una matriz y un cálculo de coste a 3 años.

> Todo se guarda en `proyecto-comunicacion/` (gráficos PNG, `dashboard.html`).

In [ ]:
import re
import shutil
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path("proyecto-comunicacion").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir()

plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10})
COLOR, GRIS, ALERTA = "#1f4e79", "#b0b7bf", "#c0392b"
print("matplotlib", matplotlib.__version__)

## 8.1 · Datos y métricas

Un año de pedidos de TiendaSur. Calculamos las métricas mensuales. Fíjate en que `ingresos_netos` se puede escribir como un **producto de factores**: eso es un árbol de métricas.

In [ ]:
# Pedidos de TiendaSur de un año (sintéticos): 120.000 pedidos de 30.000 clientes.
rng = np.random.default_rng(11)
meses = pd.period_range("2025-10", "2026-09", freq="M")
peso_mes = np.array([1.0, 1.9, 1.5, 0.9, 0.9, 1.0, 1.05, 1.1, 1.1, 1.2, 1.25, 1.3])     # noviembre: Black Friday
peso_mes = peso_mes * np.linspace(1.0, 1.35, 12)                                          # crecimiento sostenido
n = 120_000
idx = rng.choice(12, n, p=peso_mes / peso_mes.sum())
pedidos = pd.DataFrame({
    "mes": meses[idx].astype(str),
    "id_cliente": rng.integers(1, 30_001, n),
    "canal": rng.choice(["web", "tienda", "app"], n, p=[0.55, 0.30, 0.15]),
    "importe": rng.lognormal(3.3, 0.55, n).round(2),
    "devuelto": rng.random(n) < 0.06 + 0.02 * (idx == 1),                                  # más devoluciones en noviembre
    "dias_entrega": np.clip(rng.normal(3.0 + 0.8 * (idx == 1) + 0.8 * (idx == 2), 1.0, n), 1, None).round(1),
})
pedidos["margen_eur"] = pedidos["importe"] * 0.32

g = pedidos.groupby("mes")
mensual = pd.DataFrame({
    "clientes_activos": g["id_cliente"].nunique(),
    "pedidos": g.size(),
    "aov": g["importe"].mean(),                                    # valor medio del pedido (average order value)
    "tasa_devolucion": g["devuelto"].mean(),
    "dias_entrega": g["dias_entrega"].mean(),
    "margen_pct": g["margen_eur"].sum() / g["importe"].sum(),
})
mensual["pedidos_por_cliente"] = mensual["pedidos"] / mensual["clientes_activos"]
mensual["ingresos_netos"] = mensual["clientes_activos"] * mensual["pedidos_por_cliente"] * mensual["aov"] * (1 - mensual["tasa_devolucion"])
mensual.round(3)

### Métrica norte y árbol

La **métrica norte** (*north star*) es la que mejor resume el valor que recibe el cliente y que, si crece de forma sostenida, hace crecer el negocio. Para TiendaSur proponemos **ingresos netos** (ventas menos devoluciones), descompuestos en un árbol:

```
ingresos netos = clientes activos × pedidos por cliente × valor medio del pedido × (1 − tasa de devolución)
```

Cada rama la mueve un equipo distinto (captación, fidelización, catálogo y precios, calidad y logística). La descomposición logarítmica reparte la variación entre ramas **sumando exactamente 100 %**, así que puedes contestar a «¿por qué han subido las ventas?» con números.

In [ ]:
# Árbol de métricas: ingresos netos = clientes activos x pedidos/cliente x valor medio del pedido x (1 - devoluciones)
FACTORES = ["clientes_activos", "pedidos_por_cliente", "aov", "retencion_neta"]
mensual["retencion_neta"] = 1 - mensual["tasa_devolucion"]     # la parte del ingreso que "se queda"

def descomponer(mes_a, mes_b):
    """Reparte la variación de ingresos netos entre factores (descomposición logarítmica: suma exacta 100 %)."""
    a, b = mensual.loc[mes_a], mensual.loc[mes_b]
    total = np.log(b["ingresos_netos"] / a["ingresos_netos"])
    aporte = {f: np.log(b[f] / a[f]) / total for f in FACTORES}
    assert abs(sum(aporte.values()) - 1) < 1e-9                   # cuadra siempre
    return pd.Series(aporte).mul(100).round(1), b["ingresos_netos"] / a["ingresos_netos"] - 1

aportes, var = descomponer("2025-10", "2025-11")
print(f"Octubre -> noviembre: los ingresos netos cambian {var:+.1%}. Aportación de cada factor (% de la variación):")
print(aportes.to_string())
print()
aportes, var = descomponer("2026-08", "2026-09")
print(f"Agosto -> septiembre: {var:+.1%}")
print(aportes.to_string())

### Guardarraíles

Una métrica que se optimiza sola acaba haciendo daño: si solo miras ingresos, regalar el envío «funciona». Los **guardarraíles** son métricas que **no deben empeorar** mientras persigues la principal. Regla de decisión: lanzar solo si la principal mejora con evidencia **y** ningún guardarraíl se rompe.

In [ ]:
# Experimento: "envío gratis" frente a envío normal. 15.000 visitantes por grupo.
r = np.random.default_rng(5)
M = 15_000
def visitantes(prob_compra, coste_envio, dias):
    compra = r.random(M) < prob_compra
    importe = r.gamma(4, 9, M) * compra
    return pd.DataFrame({"ingreso": importe,
                         "margen": np.where(compra, importe * 0.32 - coste_envio, 0.0),
                         "dias": np.where(compra, r.normal(dias, 1.0, M), np.nan)})
control = visitantes(0.040, 0.0, 3.0)
variante = visitantes(0.047, 4.5, 3.6)          # compra más, pero el envío lo paga TiendaSur y tarda más

def ic95(a, b):
    """Diferencia de medias (variante - control) con intervalo del 95 % (aproximación normal)."""
    d = b.mean() - a.mean()
    ee = np.sqrt(a.var(ddof=1) / a.count() + b.var(ddof=1) / b.count())
    return d, d - 1.96 * ee, d + 1.96 * ee

def decidir(control, variante, principal, guardarrailes):
    """principal: columna que debe mejorar. guardarrailes: {columna: (sentido, tolerancia)}; sentido 'sube_mal' o 'baja_mal'."""
    d, lo, hi = ic95(control[principal], variante[principal])
    filas = [(principal + " (principal)", d, lo, hi, "mejora" if lo > 0 else "sin evidencia")]
    veto = []
    for col, (sentido, tol) in guardarrailes.items():
        d2, lo2, hi2 = ic95(control[col], variante[col])
        roto = (sentido == "baja_mal" and hi2 < -tol) or (sentido == "sube_mal" and lo2 > tol)
        filas.append((col + " (guardarraíl)", d2, lo2, hi2, "ROTO" if roto else "ok"))
        if roto:
            veto.append(col)
    tabla = pd.DataFrame(filas, columns=["métrica", "dif.", "IC95 inf.", "IC95 sup.", "veredicto"]).round(4)
    lanzar = lo > 0 and not veto
    return lanzar, tabla, veto

lanzar, tabla, veto = decidir(control, variante, "ingreso", {"margen": ("baja_mal", 0.0), "dias": ("sube_mal", 0.5)})
print(tabla.to_string(index=False))
print("\nDecisión:", "LANZAR" if lanzar else f"NO LANZAR (guardarraíles rotos: {', '.join(veto)})")

## 8.2 · Gráficos que convencen, y gráficos que engañan

Casi nunca se miente con datos falsos; se miente con **buenas** cifras mal dibujadas. Tres trucos habituales y cómo detectarlos: eje truncado, eje doble y ventana temporal escogida. Tufte propuso medir el primero con el **factor de mentira**: cambio visual / cambio real (honesto = 1).

In [ ]:
def factor_de_mentira(valor_a, valor_b, base_eje):
    """Factor de mentira de Tufte: cambio visual (altura de las barras desde la base del eje) / cambio real."""
    visual = ((valor_b - base_eje) - (valor_a - base_eje)) / (valor_a - base_eje)
    real = (valor_b - valor_a) / valor_a
    return visual / real

a, b = 3.10, 3.20                       # conversión (%) antes y después de un cambio en la web
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, base, titulo in [(ejes[0], 3.05, "Eje truncado: «¡se dispara!»"), (ejes[1], 0.0, "Eje desde cero: +3 %")]:
    ax.bar(["antes", "después"], [a, b], color=[GRIS, COLOR])
    ax.set_ylim(base, 3.25 if base else 3.6)
    ax.set_title(titulo)
    ax.set_ylabel("conversión (%)")
plt.tight_layout()
plt.savefig(BASE / "eje_truncado.png")
plt.show()
print(f"Factor de mentira con el eje empezando en 3,05: {factor_de_mentira(a, b, 3.05):.1f}  (honesto: 1,0)")
print(f"Factor de mentira con el eje desde cero:        {factor_de_mentira(a, b, 0.0):.1f}")

In [ ]:
# Dos engaños clásicos más: eje doble que fabrica una correlación y ventana temporal escogida a conveniencia.
mes_idx = np.arange(12)
visitas = mensual["pedidos"].to_numpy()
devol = mensual["tasa_devolucion"].to_numpy() * 100
fig, ejes = plt.subplots(1, 3, figsize=(12, 3.2))

ax = ejes[0]; ax2 = ax.twinx()
ax.plot(mes_idx, visitas, color=COLOR); ax2.plot(mes_idx, devol, color=ALERTA)
ax.set_title("Eje doble: «¡coinciden!»"); ax.set_xticks([])

serie = mensual["ingresos_netos"].to_numpy() / 1e3
ejes[1].plot(mes_idx[-3:], serie[-3:], color=COLOR, marker="o")
ejes[1].set_title("3 meses: «¡sube sin parar!»"); ejes[1].set_xticks([])

ejes[2].plot(mes_idx, serie, color=COLOR, marker="o"); ejes[2].plot(mes_idx[-3:], serie[-3:], color=ALERTA, marker="o")
ejes[2].set_title("Serie completa: hay estacionalidad"); ejes[2].set_xticks([])
for ax_ in (ejes[1], ejes[2]):
    ax_.set_ylabel("ingresos netos (miles €)")
plt.tight_layout(); plt.savefig(BASE / "enganos.png"); plt.show()

# Dos series cualquiera con tendencia parecen correlacionadas aunque no tengan nada que ver.
x, y = np.cumsum(rng.normal(size=60)), np.cumsum(rng.normal(size=60))
print(f"Correlación entre dos paseos aleatorios independientes: {np.corrcoef(x, y)[0, 1]:+.2f}  (sin relación real)")

## 8.3 · Storytelling: un gráfico, una idea

Un buen gráfico de comunicación (a diferencia de uno de exploración) tiene: **título que es la conclusión**, un único elemento resaltado (color de énfasis frente a grises), anotaciones directas en lugar de leyenda y nada que no apoye el mensaje.

In [ ]:
def grafico_historia(destino):
    """Un gráfico que cuenta algo: título = conclusión, un color de énfasis, anotación directa, sin leyenda."""
    por_canal = pedidos.groupby(["mes", "canal"])["importe"].sum().unstack() / 1e3
    fig, ax = plt.subplots(figsize=(8, 3.8))
    for canal in por_canal:
        enfasis = canal == "web"
        ax.plot(range(12), por_canal[canal], color=COLOR if enfasis else GRIS, lw=2.8 if enfasis else 1.6)
        ax.annotate(canal, (11, por_canal[canal].iloc[-1]), xytext=(6, 0), textcoords="offset points",
                    va="center", color=COLOR if enfasis else "#6b7280", fontweight="bold" if enfasis else "normal")
    pico = por_canal["web"].idxmax()
    ax.annotate("Black Friday", (list(por_canal.index).index(pico), por_canal["web"].max()),
                xytext=(3, por_canal["web"].max() * 0.97), arrowprops={"arrowstyle": "->", "color": "#6b7280"}, color="#6b7280")
    ax.set_xticks(range(12)); ax.set_xticklabels([m[2:] for m in por_canal.index], rotation=0, fontsize=8)
    ax.set_ylabel("ingresos brutos (miles €)")
    total = por_canal.sum(axis=1)
    cuota_web = por_canal["web"].sum() / total.sum()
    ax.set_title(f"La web aporta el {cuota_web:.0%} de las ventas; Black Friday\nmultiplicó por {total['2025-11'] / total['2025-10']:.1f} las de octubre", loc="left")
    plt.tight_layout(); plt.savefig(destino); plt.show()

grafico_historia(BASE / "historia.png")
print("Comprueba: ¿el título dice qué hacer con el gráfico sin leerlo entero? ¿Hay una sola cosa resaltada?")

## 8.4 · Un dashboard sencillo

Principios: responde a **una pregunta** de **una persona** concreta; lo más importante arriba a la izquierda; pocos números grandes con su comparación; el guardarraíl siempre visible con su umbral; y nada decorativo. Lo construimos con Plotly y se guarda como un HTML autocontenido (necesita internet para cargar la librería desde un CDN).

In [ ]:
def construir_dashboard(destino):
    """Un dashboard sencillo: KPIs arriba, tendencia, desglose y guardarraíl con su umbral."""
    ult, pen = mensual.iloc[-1], mensual.iloc[-2]
    fig = make_subplots(
        rows=3, cols=4, row_heights=[0.18, 0.41, 0.41],
        specs=[[{"type": "domain"}] * 4, [{"colspan": 4, "type": "xy"}, None, None, None],
               [{"colspan": 2, "type": "xy"}, None, {"colspan": 2, "type": "xy"}, None]],
        subplot_titles=("", "", "", "", "Ingresos netos mensuales (miles €)",
                        "Ingresos brutos por canal, último mes (miles €)", "Guardarraíl: tasa de devolución (%)"))
    kpis = [("Ingresos netos (miles €)", ult["ingresos_netos"] / 1e3, pen["ingresos_netos"] / 1e3, ".0f"),
            ("Clientes activos", ult["clientes_activos"], pen["clientes_activos"], ",.0f"),
            ("Valor medio del pedido (€)", ult["aov"], pen["aov"], ".1f"),
            ("Devoluciones (%)", ult["tasa_devolucion"] * 100, pen["tasa_devolucion"] * 100, ".1f")]
    for i, (titulo, v, ref, fmt) in enumerate(kpis, start=1):
        fig.add_trace(go.Indicator(mode="number+delta", value=v, title={"text": titulo, "font": {"size": 13}},
                                   number={"valueformat": fmt}, delta={"reference": ref, "relative": True, "valueformat": ".1%",
                                   "increasing": {"color": "#c0392b" if i == 4 else "#2e7d32"},
                                   "decreasing": {"color": "#2e7d32" if i == 4 else "#c0392b"}}), row=1, col=i)
    fig.add_trace(go.Scatter(x=list(mensual.index), y=mensual["ingresos_netos"] / 1e3, mode="lines+markers",
                             line={"color": COLOR}, name="ingresos netos"), row=2, col=1)
    ultimo_mes = pedidos[pedidos["mes"] == mensual.index[-1]].groupby("canal")["importe"].sum() / 1e3
    fig.add_trace(go.Bar(x=ultimo_mes.index, y=ultimo_mes.values, marker_color=COLOR, name="canal"), row=3, col=1)
    fig.add_trace(go.Scatter(x=list(mensual.index), y=mensual["tasa_devolucion"] * 100, mode="lines+markers",
                             line={"color": ALERTA}, name="devoluciones"), row=3, col=3)
    fig.add_trace(go.Scatter(x=[mensual.index[0], mensual.index[-1]], y=[7.0, 7.0], mode="lines+text", text=["", "umbral acordado: 7 %"],
                             textposition="top left", line={"color": "#6b7280", "dash": "dash"}), row=3, col=3)
    fig.update_layout(height=720, showlegend=False, template="simple_white", margin={"t": 50, "b": 30},
                      title={"text": "TiendaSur · Resumen mensual (datos hasta " + mensual.index[-1] + ")", "x": 0.01})
    fig.write_html(destino, include_plotlyjs="cdn")      # un único archivo; se abre en cualquier navegador
    return fig

dash = construir_dashboard(BASE / "dashboard.html")
html = (BASE / "dashboard.html").read_text()
print(f"dashboard.html: {len(html) / 1e3:.0f} kB · {len(dash.data)} elementos · incluye datos de {mensual.index[-1]}: {mensual.index[-1] in html}")

## 8.5 · Escribir para ejecutivos: el memo de una página

**Pirámide de Minto:** primero la conclusión (la recomendación y lo que pides), después los argumentos, al final el detalle. El directivo que solo lea el primer párrafo debe poder decidir. Una página, sin jerga, con números concretos y con la decisión y la fecha.

In [ ]:
PIRAMIDE = """\
MEMO · {asunto}
Para: {para}   De: {de}   Fecha: {fecha}

RECOMENDACIÓN
{recomendacion}

POR QUÉ (tres razones, con números)
{razones}

RIESGOS Y CÓMO LOS CONTROLAMOS
{riesgos}

DECISIÓN QUE NECESITO
{decision}
"""

def redactar_memo(datos):
    razones = "\n".join(f"{i}. {t}" for i, t in enumerate(datos["razones"], start=1))
    riesgos = "\n".join(f"- {t}" for t in datos["riesgos"])
    return PIRAMIDE.format(**{**datos, "razones": razones, "riesgos": riesgos})

def revisar_memo(texto, max_palabras=300):
    """Comprobaciones mecánicas: longitud, recomendación al principio y jerga."""
    palabras = len(texto.split())
    pos_reco = texto.find("RECOMENDACIÓN")
    primeras = " ".join(texto.split()[:60])
    jerga = [j for j in ["p-valor", "p valor", "significativ", "desviación típica", "regresión", "intervalo de confianza", "hiperparámetro"] if j in texto.lower()]
    return {"palabras": palabras, "cabe_en_una_pagina": palabras <= max_palabras,
            "recomendacion_arriba": pos_reco != -1 and "RECOMENDACIÓN" in primeras, "jerga": jerga}

d_ing = ic95(control["ingreso"], variante["ingreso"])[0]
d_mar = ic95(control["margen"], variante["margen"])[0]
d_dias = ic95(control["dias"], variante["dias"])[0]
d_compras = ((variante["ingreso"] > 0).mean() - (control["ingreso"] > 0).mean()) * 1000
fmt = lambda x, d=1: f"{x:+.{d}f}".replace(".", ",")          # número con coma decimal y signo

memo = redactar_memo({
    "asunto": "Envío gratis para todos los pedidos", "para": "Dirección comercial", "de": "Equipo de datos", "fecha": "23/10/2026",
    "recomendacion": "No lanzar el envío gratis a todos los clientes. Sí probarlo con un mínimo de compra de 40 €, que protege el margen.",
    "razones": [
        f"Vende más: {fmt(d_compras, 0)} compras por cada 1.000 visitantes y {fmt(d_ing, 2)} € de ingresos por visitante en la prueba.",
        f"Pero gana menos: el margen por visitante cambia {fmt(d_mar, 2)} € porque TiendaSur paga unos 4,50 € de envío por pedido.",
        f"Entrega más tarde: {fmt(d_dias, 1)} días de media, justo cuando el plazo de entrega es nuestra principal queja.",
    ],
    "riesgos": ["Con un mínimo de compra puede que se venda algo menos: lo medimos en una segunda prueba de 3 semanas.",
                "Si la prueba se alarga, hay riesgo de efecto novedad: revisamos la semana 1 frente a la 3."],
    "decision": "Aprobar la prueba con mínimo de 40 € antes del 6 de noviembre, para tener resultados antes de Navidad.",
})
print(memo)
print(revisar_memo(memo))

## 8.6 · Presentar a público no técnico

Reale, ICF y casi cualquier empresa que contrata perfiles de datos piden «comunicar resultados a perfiles no técnicos». En la práctica significa: **frecuencias naturales** («28 de cada 1.000») en lugar de porcentajes y probabilidades, **incertidumbre en una frase** en lugar de «p < 0,05», y una **sola decisión** al final.

In [ ]:
def traducir_ab(p_control, p_variante, n_por_grupo, por=1000):
    """Convierte un resultado de A/B en una frase con frecuencias naturales, sin jerga."""
    d = p_variante - p_control
    ee = np.sqrt(p_control * (1 - p_control) / n_por_grupo + p_variante * (1 - p_variante) / n_por_grupo)
    lo, hi = d - 1.96 * ee, d + 1.96 * ee
    if lo > 0:
        veredicto = "La mejora es real con bastante seguridad."
    elif hi < 0:
        veredicto = "Empeora con bastante seguridad."
    else:
        veredicto = "Con estos datos no podemos asegurar que haya diferencia: habría que probar más tiempo."
    miles = f"{por:,}".replace(",", ".")
    return (f"De cada {miles} visitantes, {p_variante * por:.0f} compran con la versión nueva y {p_control * por:.0f} con la actual. "
            f"La diferencia real podría estar entre {lo * por:+.0f} y {hi * por:+.0f} compras por cada {miles}. {veredicto}")

print(traducir_ab(0.040, 0.047, 15_000))
print(traducir_ab(0.040, 0.042, 3_000))
print("\nCompara con la versión técnica: «diferencia de proporciones 0,007, IC95 [0,002; 0,012], p = 0,003».")

## 8.7 · Make or buy

¿Construimos el panel de ventas a medida o compramos una herramienta de BI? Dos herramientas: una **matriz ponderada** (que explicita los criterios y sus pesos, para discutir los pesos y no las opiniones) y un **TCO** (coste total de propiedad) con una **sensibilidad** a la hipótesis que más pesa.

In [ ]:
# Make or buy: matriz ponderada + coste total a 3 años (TCO).
criterios = pd.DataFrame({
    "peso":  [0.30, 0.20, 0.15, 0.15, 0.10, 0.10],
    "construir": [5, 3, 2, 5, 2, 5],            # puntuación 1-5 (5 = mejor)
    "comprar":   [3, 5, 5, 3, 5, 2],
}, index=["Encaje con nuestras necesidades", "Tiempo hasta tener valor", "Mantenimiento/operación", "Control y flexibilidad",
          "Riesgo de dependencia del proveedor", "Control de datos y cumplimiento"])
assert abs(criterios["peso"].sum() - 1) < 1e-9
puntos = (criterios[["construir", "comprar"]].mul(criterios["peso"], axis=0)).sum()
print(criterios)
print("\nPuntuación ponderada:", puntos.round(2).to_dict())

def tco(usuarios, años=3, coste_persona_mes=5_500, meses_construir=4, mantenimiento_anual=0.20, infra_anual=6_000,
        licencia_usuario_mes=25, implantacion=8_000):
    construir = coste_persona_mes * meses_construir * (1 + mantenimiento_anual * (años - 1)) + infra_anual * años
    comprar = licencia_usuario_mes * 12 * usuarios * años + implantacion
    return construir, comprar

filas = []
for u in [10, 25, 50, 100, 200]:
    c, b = tco(u)
    filas.append((u, round(c), round(b), "construir" if c < b else "comprar"))
sens = pd.DataFrame(filas, columns=["usuarios", "TCO construir (€)", "TCO comprar (€)", "más barato"])
print("\nSensibilidad al número de usuarios (3 años):")
print(sens.to_string(index=False))

## Qué has construido

Un conjunto de piezas de comunicación reutilizables: descomposición de una métrica, regla de decisión con guardarraíles, gráficos honestos, un dashboard HTML, un memo revisable automáticamente y un análisis make-or-buy. En el caso práctico las unirás en un **informe-producto** sobre un conjunto de datos propio.

## Ejercicios

1. **Árbol de métricas.** Descompón la variación de ingresos netos entre `2026-07` y `2026-08` con `descomponer`. Escribe en una frase, para el director, qué factor explica más y qué harías.
2. **Factor de mentira.** Dibuja las ventas de dos tiendas (`120` y `135` mil €) con un eje que empiece en `100`. Calcula su factor de mentira y rehaz el gráfico para que valga entre 0,95 y 1,05.
3. **Guardarraíles.** Simula un experimento de *cupón del 10 %* (más compras, pero margen por pedido más bajo y más devoluciones) y usa `decidir` con tres guardarraíles. ¿Lanzarías? Justifica con la tabla.
4. **Dashboard.** Amplía el panel del guardarraíl de `construir_dashboard` con la tasa de devolución mensual de cada canal (una línea fina por canal, sobre el mismo umbral). Guarda `dashboard_v2.html` y comprueba que tiene 3 elementos más que el original.
5. **Memo sin jerga.** Redacta con `redactar_memo` una recomendación sobre el resultado del ejercicio 3 (máximo 250 palabras). Debe pasar `revisar_memo` con `cabe_en_una_pagina=True`, `recomendacion_arriba=True` y sin jerga.

In [ ]:
# Ejercicio 1 · Julio -> agosto
aportes, var = descomponer("2026-07", "2026-08")
print(f"Ingresos netos {var:+.1%}")
print(aportes.to_string())
principal = aportes.abs().idxmax()
print(f"\nFrase: «Los ingresos netos {'suben' if var > 0 else 'bajan'} un {abs(var):.1%} y el factor que más pesa es "
      f"{principal.replace('_', ' ')} ({aportes[principal]:.0f} % de la variación); actuaría sobre ese factor primero.»")

In [ ]:
# Ejercicio 2 · Dos tiendas con un eje que empieza en 100
a, b = 120, 135
print(f"Factor de mentira con base 100: {factor_de_mentira(a, b, 100):.1f}")
fig, ejes = plt.subplots(1, 2, figsize=(8, 3))
for ax, base in zip(ejes, (100, 0)):
    ax.bar(["Tienda A", "Tienda B"], [a, b], color=[GRIS, COLOR]); ax.set_ylim(base, 150)
    ax.set_title(f"Eje desde {base}: factor {factor_de_mentira(a, b, base):.1f}")
plt.tight_layout(); plt.show()
assert 0.95 <= factor_de_mentira(a, b, 0) <= 1.05
print("Con el eje desde 0 el factor es 1,0: la diferencia visual coincide con la real (+12,5 %).")

In [ ]:
# Ejercicio 3 · Cupón del 10 %
r = np.random.default_rng(9)
def visitantes_cupon(p, descuento, dev):
    compra = r.random(M) < p
    importe = r.gamma(4, 9, M) * compra * (1 - descuento)
    devuelto = (r.random(M) < dev) & compra
    return pd.DataFrame({"ingreso": importe, "margen": np.zeros(M),
                         "devuelto": devuelto.astype(float)})
ctrl = visitantes_cupon(0.040, 0.0, 0.06)
cupon = visitantes_cupon(0.047, 0.10, 0.09)
# El cupón reduce el precio, así que el margen por euro vendido baja: lo recalculamos explícitamente.
ctrl["margen"] = ctrl["ingreso"] * 0.32
cupon["margen"] = cupon["ingreso"] * (0.32 - 0.10) / 0.90          # mismo coste, ingreso menor
lanzar, tabla, veto = decidir(ctrl, cupon, "ingreso", {"margen": ("baja_mal", 0.0), "devuelto": ("sube_mal", 0.0005)})
print(tabla.to_string(index=False))
print("\nDecisión:", "LANZAR" if lanzar else f"NO LANZAR (guardarraíles rotos: {', '.join(veto)})")

In [ ]:
# Ejercicio 4 · Devoluciones por canal en el panel del guardarraíl
fig2 = construir_dashboard(BASE / "dashboard_v2.html")
dev_canal = pedidos.groupby(["mes", "canal"])["devuelto"].mean().unstack() * 100
for canal, color in zip(dev_canal.columns, ["#7f8c8d", "#e67e22", "#8e44ad"]):
    fig2.add_trace(go.Scatter(x=list(dev_canal.index), y=dev_canal[canal], mode="lines", line={"color": color, "width": 1},
                              name=f"devoluciones {canal}"), row=3, col=3)
fig2.write_html(BASE / "dashboard_v2.html", include_plotlyjs="cdn")
print("Elementos: original", len(dash.data), "-> v2", len(fig2.data))
assert len(fig2.data) == len(dash.data) + 3
print("Aviso de diseño: más líneas = más ruido. Si no cambian la decisión, déjalas fuera o ponlas en una vista de detalle.")

In [ ]:
# Ejercicio 5 · Memo sobre el cupón del 10 %
memo5 = redactar_memo({
    "asunto": "Cupón del 10 % en la próxima campaña", "para": "Dirección comercial", "de": "Equipo de datos", "fecha": "23/10/2026",
    "recomendacion": "No lanzar el cupón del 10 % a toda la base de clientes: consigue más compras, pero deja menos beneficio.",
    "razones": ["Más compras, pero no más ingresos: 47 por cada 1.000 visitantes frente a 40, con un 10 % menos de ingreso por pedido.",
                "Gana menos: el margen por visitante baja unos 0,12 € porque cada pedido deja menos beneficio.",
                "Devuelven más: las devoluciones pasan del 6 % al 9 % de los pedidos."],
    "riesgos": ["Podemos perder ventas si no hacemos nada: proponemos un cupón solo para clientes inactivos."],
    "decision": "Aprobar una prueba con clientes inactivos durante dos semanas.",
})
print(memo5)
res = revisar_memo(memo5, max_palabras=250)
print(res)
assert res["cabe_en_una_pagina"] and res["recomendacion_arriba"] and not res["jerga"]